# Imports

In [1]:
import pandas as pd
import glob
import os
import requests
import warnings
warnings.filterwarnings("ignore")

# Load all 24 snapshots:

In [2]:
folder = r"..\..\Data\Main\Quarterly"
files = sorted(glob.glob(os.path.join(folder, "SPX as of*.xlsx")))
print(f"Found {len(files)} snapshot files")

all_snapshots = []
for f in files:
    fname = os.path.basename(f)
    snapshot = pd.read_excel(f)
    
    date_str = fname.replace("SPX as of ", "").replace("1.xlsx", "").strip()
    quarter_date = pd.to_datetime(date_str, format="%b %d %Y")
    
    snapshot['quarter_date'] = quarter_date
    all_snapshots.append(snapshot)
    print(f"  {fname}: {len(snapshot)} tickers, date: {quarter_date.date()}")

combined = pd.concat(all_snapshots, ignore_index=True)
print(f"\nCombined: {len(combined)} rows")

Found 24 snapshot files
  SPX as of Apr 01 20201.xlsx: 506 tickers, date: 2020-04-01
  SPX as of Apr 01 20211.xlsx: 505 tickers, date: 2021-04-01
  SPX as of Apr 01 20221.xlsx: 506 tickers, date: 2022-04-01
  SPX as of Apr 01 20241.xlsx: 504 tickers, date: 2024-04-01
  SPX as of Apr 01 20251.xlsx: 503 tickers, date: 2025-04-01
  SPX as of Apr 03 20231.xlsx: 503 tickers, date: 2023-04-03
  SPX as of Jan 02 20201.xlsx: 505 tickers, date: 2020-01-02
  SPX as of Jan 02 20241.xlsx: 503 tickers, date: 2024-01-02
  SPX as of Jan 02 20251.xlsx: 503 tickers, date: 2025-01-02
  SPX as of Jan 03 20221.xlsx: 505 tickers, date: 2022-01-03
  SPX as of Jan 03 20231.xlsx: 503 tickers, date: 2023-01-03
  SPX as of Jan 04 20211.xlsx: 505 tickers, date: 2021-01-04
  SPX as of Jul 01 20201.xlsx: 505 tickers, date: 2020-07-01
  SPX as of Jul 01 20211.xlsx: 506 tickers, date: 2021-07-01
  SPX as of Jul 01 20221.xlsx: 503 tickers, date: 2022-07-01
  SPX as of Jul 01 20241.xlsx: 503 tickers, date: 2024-07-01


# Clean tickers and identify Bloomberg IDs

In [3]:
def clean_ticker(raw):
    ticker = raw.split(" ")[0]
    ticker = ticker.replace("/", "-")
    return ticker

combined['yf_ticker'] = combined['Ticker'].apply(clean_ticker)

bloomberg_ids = combined[combined['yf_ticker'].str.match(r'^\d')][['Ticker', 'Name', 'yf_ticker']].drop_duplicates()
print(f"Bloomberg internal IDs found: {len(bloomberg_ids)}\n")
print(bloomberg_ids.to_string(index=False))

Bloomberg internal IDs found: 7

            Ticker                           Name yf_ticker
2078185D UN Equity                   Tiffany & Co  2078185D
2499073D UN Equity                 IHS Markit Ltd  2499073D
9903115D UN Equity          Blackrock Finance Inc  9903115D
9980328D UN Equity Laboratory Corp of America Hol  9980328D
9983490D UN Equity            TE Connectivity Ltd  9983490D
9990213D UN Equity   Jacobs Engineering Group Inc  9990213D
9991429D UN Equity       Aptiv Irish Holdings Ltd  9991429D


# Attempt to resolve Bloomberg IDs via Yahoo Finance search

In [4]:
bloomberg_ids_list = bloomberg_ids[['yf_ticker', 'Name']].values.tolist()

print("Attempting to resolve Bloomberg IDs via Yahoo Finance search:\n")
resolved = {}
unresolved = {}

for bbg_id, name in bloomberg_ids_list:
    try:
        url = f"https://query2.finance.yahoo.com/v1/finance/search?q={name}&quotesCount=5"
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        results = response.json().get('quotes', [])
        
        if results:
            top = results[0]
            print(f"{bbg_id} ({name})")
            print(f"  → Best match: {top['symbol']} ({top.get('longname', 'N/A')})")
            resolved[bbg_id] = {
                'name': name,
                'suggested_ticker': top['symbol'],
                'match_name': top.get('longname', 'N/A')
            }
        else:
            print(f"{bbg_id} ({name}) → No results found")
            unresolved[bbg_id] = name
    except Exception as e:
        print(f"{bbg_id} ({name}) → Error: {e}")
        unresolved[bbg_id] = name

print(f"\nResolved: {len(resolved)}, Unresolved: {len(unresolved)}")

Attempting to resolve Bloomberg IDs via Yahoo Finance search:

2078185D (Tiffany & Co) → No results found
2499073D (IHS Markit Ltd) → No results found
9903115D (Blackrock Finance Inc)
  → Best match: BLQ.F (BlackRock, Inc.)
9980328D (Laboratory Corp of America Hol) → No results found
9983490D (TE Connectivity Ltd) → No results found
9990213D (Jacobs Engineering Group Inc)
  → Best match: Z0Y.SG (Jacobs Engineering Group Inc)
9991429D (Aptiv Irish Holdings Ltd) → No results found

Resolved: 2, Unresolved: 5


## Retry with Simplified Search Queries

In [5]:
# The full Bloomberg names didn't work well. Trying shorter/simpler names
# Build retry queries using shorter/simpler versions of the company names
# IDs come from Cell 3's bloomberg_ids dataframe
retry_queries = {}
for _, row in bloomberg_ids.iterrows():
    bbg_id = row['yf_ticker']
    name = row['Name'].split(" ")[0]  # Take first word as simplified query
    retry_queries[bbg_id] = name

# Override where first word alone isn't enough
retry_queries['2499073D'] = 'IHS Markit'
retry_queries['9983490D'] = 'TE Connectivity'
retry_queries['9990213D'] = 'Jacobs Engineering'
retry_queries['9980328D'] = 'LabCorp'

print("Retry queries:")
for k, v in retry_queries.items():
    print(f"  {k} → '{v}'")



print("Retrying with simplified search queries:\n")
for bbg_id, query in retry_queries.items():
    try:
        url = f"https://query2.finance.yahoo.com/v1/finance/search?q={query}&quotesCount=5"
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        results = response.json().get('quotes', [])
        
        print(f"{bbg_id} (query: '{query}'):")
        if results:
            for r in results[:3]:
                exchange = r.get('exchange', 'N/A')
                print(f"  {r['symbol']} — {r.get('longname', r.get('shortname', 'N/A'))} [{exchange}]")
        else:
            print("  No results")
        print()
    except Exception as e:
        print(f"  Error: {e}\n")

Retry queries:
  2078185D → 'Tiffany'
  2499073D → 'IHS Markit'
  9903115D → 'Blackrock'
  9980328D → 'LabCorp'
  9983490D → 'TE Connectivity'
  9990213D → 'Jacobs Engineering'
  9991429D → 'Aptiv'
Retrying with simplified search queries:

2078185D (query: 'Tiffany'):
  No results

2499073D (query: 'IHS Markit'):
  400590.KS — Shinhan Asset Management - SOL IHS Markit Global Carbon Emission(Synthetic) ETF [KSC]

9903115D (query: 'Blackrock'):
  BLK — BlackRock, Inc. [NYQ]
  HYT — BlackRock Corporate High Yield Fund, Inc. [NYQ]
  BRC.V — Blackrock Silver Corp. [VAN]

9980328D (query: 'LabCorp'):
  LH — Labcorp Holdings Inc. [NYQ]
  N6B.F — Labcorp Holdings Inc. [FRA]
  0JSY.L — Labcorp Holdings Inc. [LSE]

9983490D (query: 'TE Connectivity'):
  TEL — TE Connectivity plc [NYQ]
  BZ4.F — TE Connectivity plc [FRA]
  T1EL34.SA — TE Connectivity plc [SAO]

9990213D (query: 'Jacobs Engineering'):
  Z0Y.SG — Jacobs Engineering Group Inc [STU]

9991429D (query: 'Aptiv'):
  APTV — Aptiv PLC [NYQ

In [6]:
print("4 resolved to US tickers: BLK, LH, TEL, APTV")
print("1 partially resolved (Jacobs) — US ticker needs manual identification")
print("2 unresolved (Tiffany, IHS Markit) — need Bloomberg Terminal lookup")

4 resolved to US tickers: BLK, LH, TEL, APTV
1 partially resolved (Jacobs) — US ticker needs manual identification
2 unresolved (Tiffany, IHS Markit) — need Bloomberg Terminal lookup


## Review Yahoo Finance Results — 4 Resolved, 1 Partial, 2 Unresolved

 Retry with simplified queries resolved 4 to US-listed tickers:
- 9903115D (Blackrock Finance Inc) → BLK — BlackRock, Inc. [NYQ] ✓
- 9980328D (Laboratory Corp of America) → LH — Labcorp Holdings Inc. [NYQ] ✓(needed "LabCorp" as query — full Bloomberg name returned irrelevant results)
- 9983490D (TE Connectivity Ltd) → TEL — TE Connectivity plc [NYQ] ✓
- 9991429D (Aptiv Irish Holdings Ltd) → APTV — Aptiv PLC [NYQ] ✓

1 partially resolved (non-US exchange only):
- 9990213D (Jacobs Engineering) → Z0Y.SG [Stuttgart only]
- US ticker needs further investigation.
Need to manually identify the US ticker.

2 unresolved (likely delisted):
- 2078185D (Tiffany & Co) → No results at all
- 2499073D (IHS Markit) → Only returned an unrelated Korean ETF
Both need Bloomberg Terminal verification using DES command.



## Resolve Jacobs Engineering — Company Rebranded to Jacobs Solutions

In [7]:
# Jacobs Engineering — Yahoo Finance only returned non-US exchanges.
# Searching Google for the US-listed ticker reveals the company rebranded 
# to Jacobs Solutions Inc. 
# Trying that name:

query = 'Jacobs Solutions'
url = f"https://query2.finance.yahoo.com/v1/finance/search?q={query}&quotesCount=5"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
results = response.json().get('quotes', [])

print(f"Query: '{query}':")
for r in results[:3]:
    exchange = r.get('exchange', 'N/A')
    print(f"  {r['symbol']} — {r.get('longname', r.get('shortname', 'N/A'))} [{exchange}]")

Query: 'Jacobs Solutions':
  J — Jacobs Solutions Inc. [NYQ]
  0JOI.L — Jacobs Solutions Inc. [LSE]
  Z0Y.F — Jacobs Solutions Inc. [FRA]


# Apply Confirmed Ticker Mappings and Exclude Acquired Companies

In [8]:
bloomberg_to_ticker = {
    '9903115D': 'BLK',
    '9980328D': 'LH',
    '9983490D': 'TEL',
    '9990213D': 'J',
    '9991429D': 'APTV',
}

combined['yf_ticker'] = combined['yf_ticker'].replace(bloomberg_to_ticker)
combined = combined[~combined['yf_ticker'].isin(['2078185D', '2499073D'])]

print(f"Resolved 5 Bloomberg IDs to real tickers: {list(bloomberg_to_ticker.values())}")
print(f"Excluded 2 acquired companies: 2078185D (Tiffany), 2499073D (IHS Markit)")
print(f"Remaining rows: {len(combined)}")
print(f"Unique tickers: {combined['yf_ticker'].nunique()}")

Resolved 5 Bloomberg IDs to real tickers: ['BLK', 'LH', 'TEL', 'J', 'APTV']
Excluded 2 acquired companies: 2078185D (Tiffany), 2499073D (IHS Markit)
Remaining rows: 12086
Unique tickers: 606


# Build Membership Panel (first_seen, last_seen, quarters_present)

The 24 snapshots cover Q1 2020 to Q4 2025 using the first business day of each quarter as the reference date (Jan, Apr, Jul, Oct). The last snapshot is October 1, 2025. A January 2026 snapshot was not taken because the study period ends in December 2025 — a January 2026 snapshot would reflect index changes that occurred after the study window. Additionally, any tickers that joined the index in Q4 2025 would only have approximately 60 trading days of data, which falls below the 100 non-zero sentiment day minimum applied later in the pipeline. Tickers present in the October 2025 snapshot are assumed to remain in the index through year-end, so their last_seen is extended from 2025-10-01 to 2025-12-31 to ensure full data coverage for Q4 2025.

In [9]:
panel = combined.groupby('yf_ticker').agg(
    name=('Name', 'first'),
    first_seen=('quarter_date', 'min'),
    last_seen=('quarter_date', 'max'),
    quarters_present=('quarter_date', 'nunique')
).reset_index()

latest_snapshot = combined['quarter_date'].max()
print(f"Latest snapshot date: {latest_snapshot.date()}")

# Extend to end of the quarter that the last snapshot belongs to
quarter_end = latest_snapshot + pd.offsets.QuarterEnd(0)
print(f"Extending active tickers' last_seen from {latest_snapshot.date()} to {quarter_end.date()}")
panel.loc[panel['last_seen'] == latest_snapshot, 'last_seen'] = quarter_end

print(f"\nPanel shape: {panel.shape}")
print(f"Tickers present all 24 quarters: {(panel['quarters_present'] == 24).sum()}")
print(f"\nSample:")
print(panel.head(10).to_string(index=False))

Latest snapshot date: 2025-10-01
Extending active tickers' last_seen from 2025-10-01 to 2025-12-31

Panel shape: (606, 5)
Tickers present all 24 quarters: 417

Sample:
yf_ticker                        name first_seen  last_seen  quarters_present
        A    Agilent Technologies Inc 2020-01-02 2025-12-31                24
      AAL American Airlines Group Inc 2020-01-02 2024-07-01                19
      AAP      Advance Auto Parts Inc 2020-01-02 2023-07-03                15
     AAPL                   Apple Inc 2020-01-02 2025-12-31                24
     ABBV                  AbbVie Inc 2020-01-02 2025-12-31                24
     ABMD                 ABIOMED Inc 2020-01-02 2022-10-03                12
     ABNB                  Airbnb Inc 2023-10-02 2025-12-31                 9
      ABT         Abbott Laboratories 2020-01-02 2025-12-31                24
     ACGL      Arch Capital Group Ltd 2023-01-03 2025-12-31                12
      ACN               Accenture PLC 2020-01-02 202

# Save Final Panel and Combined Quarterly Data

In [10]:
panel.to_csv(r"..\..\Data\Main\membership_panel_clean.csv", index=False)
combined.to_csv(r"..\..\Data\Main\combined_quarterly.csv", index=False)

print("Saved: membership_panel_clean.csv")
print("Saved: combined_quarterly.csv")
print(f"\nFinal panel: {len(panel)} unique tickers")
print(f"Combined quarterly: {len(combined)} rows")

Saved: membership_panel_clean.csv
Saved: combined_quarterly.csv

Final panel: 606 unique tickers
Combined quarterly: 12086 rows
